# Benchmark: WebDataset-style shards (tar streaming over S3)


In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import webdataset as wds
import fsspec

print('torch:', torch.__version__)


## Configuration

In [ ]:
S3_BUCKET = os.environ.get('S3_BUCKET', '')
S3_PREFIX = os.environ.get('S3_PREFIX', 'Food-11-webdataset')
SPLIT = 'training'
S3_ENDPOINT_URL = os.environ.get('S3_ENDPOINT_URL', '')

BATCH_SIZE = 64
NUM_WORKERS = 8

WARMUP_BATCHES = 10
MEASURE_BATCHES = 200


print('S3_BUCKET:', S3_BUCKET)
print('S3_PREFIX:', S3_PREFIX)
print('SPLIT:', SPLIT)
print('S3_ENDPOINT_URL:', S3_ENDPOINT_URL if S3_ENDPOINT_URL else '(default)')
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS:', NUM_WORKERS)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)



## Dataset (WebDataset pipeline over tar shards)

In [ ]:
fs_list = fsspec.filesystem(
    's3',
    client_kwargs={'endpoint_url': S3_ENDPOINT_URL},
)

shard_glob = f"{S3_BUCKET}/{S3_PREFIX}/{SPLIT}/*.tar"
shards = fs_list.glob(shard_glob)
shards = [s for s in shards if not s.endswith('/')]
shards.sort()

if not shards:
    raise FileNotFoundError(f'No shards found at: s3://{shard_glob}')

print('num_shards:', len(shards))
print('first_shard:', shards[0])

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

from webdataset.tariterators import tar_file_expander, group_by_keys

_FS = None

def get_fs():
    global _FS
    if _FS is None:
        _FS = fsspec.filesystem('s3', client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
    return _FS

def add_stream(sample):
    # sample: {'url': <shard_path>}
    fs = get_fs()
    url = sample['url']
    sample['stream'] = fs.open(url, 'rb')
    return sample

dataset = wds.DataPipeline(
    wds.SimpleShardList(shards),
    wds.split_by_worker,
    wds.map(add_stream),
    tar_file_expander,
    group_by_keys,
    wds.decode('pil'),
    wds.to_tuple('jpg', 'cls'),
    wds.map_tuple(transform, lambda b: int(b)),
)


## DataLoader

In [ ]:
num_workers = NUM_WORKERS
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    drop_last=False,
    prefetch_factor=2,
    persistent_workers=True,
)


## Run benchmark

In [ ]:
it = iter(loader)

for _ in range(WARMUP_BATCHES):
    try:
        _ = next(it)
    except StopIteration:
        break

num_batches = 0
num_items = 0
t_start = time.perf_counter()
for _ in range(MEASURE_BATCHES):
    try:
        x, y = next(it)
    except StopIteration:
        break
    num_batches += 1
    num_items += int(y.shape[0])
t_end = time.perf_counter()

wall_s = t_end - t_start
imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

result = {
    'num_workers': num_workers,
    'batch_size': BATCH_SIZE,
    'measured_batches': num_batches,
    'measured_items': num_items,
    'wall_s': wall_s,
    'imgs_per_s': imgs_per_s,
    'batches_per_s': batches_per_s,
    'avg_batch_s': avg_batch_s,
}

result


## Print results

In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

avg_batch_s = result['avg_batch_s']
avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
print(
    'workers=', result['num_workers'],
    'imgs/s=', f"{result['imgs_per_s']:.2f}",
    'batches/s=', f"{result['batches_per_s']:.2f}",
    'avg_batch_s=', avg_batch_s_str,
)


## Save results

In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'webdataset_{stamp}.json'
payload = {
    'benchmark': 'webdataset',
    'timestamp_utc': stamp,
    's3_bucket': S3_BUCKET,
    's3_prefix': S3_PREFIX,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'result': result,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
